# 声帯音源モデルの比較研究

## なぜ音源波形が重要か

声道フィルタ（波動管）がどれだけ精密でも、入力する声帯音源の波形が不適切であれば  
自然な声は得られない。音源波形は **声質・音色の大半を決定する** 要素である。

### 声帯音源の物理的意味

```
肺（気圧）→ 声帯（バルブ）→ グロタルフロー Ug(t) → 声道フィルタ → 口唇放射 → 音声
```

声道フィルタへの入力は **グロタルフロー Ug(t)** または  
その微分 **dUg/dt**（急激な声帯閉鎖が主要な励振になる）。

**声帯閉鎖の瞬間**（glottal closure instant）に大きな負のスパイクが発生し、  
これが声道フォルマントを励振する「バズ（buzz）」の正体である。

### 比較するモデル

| モデル | 種別 | 特徴 | 複雑さ |
|--------|------|------|--------|
| **のこぎり波** | 現在実装中 | 簡易・非物理的 | ★☆☆ |
| **Rosenberg C パルス** | フロー近似 | 区分多項式・滑らか | ★★☆ |
| **LF モデル** | フロー微分 | 最も物理的・パラメトリック | ★★★ |

**参考文献**
- Rosenberg, A.E. (1971). Effect of glottal pulse shape on the quality of natural vowels. *JASA*, 49(2B), 583–590.
- Fant, G., Liljencrants, J., & Lin, Q. (1985). A four-parameter model of glottal flow. *STL-QPSR*, 4, 1–13. (LF モデル)
- Fant, G. (1995). The LF-model revisited. *STL-QPSR*, 36(2-3), 119–156.
- de Boor et al. (2003). On the LF model of the glottal source. *Fonetik*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from IPython.display import Audio, display
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#1e293b', 'axes.facecolor': '#0f172a',
    'axes.edgecolor':   '#475569', 'axes.labelcolor': '#cbd5e1',
    'xtick.color':      '#94a3b8', 'ytick.color':     '#94a3b8',
    'text.color':       '#e2e8f0', 'grid.color':       '#334155',
    'grid.linestyle':   '--',      'grid.alpha':        0.6,
    'lines.linewidth':  1.8,
})

FS = 44100
print('準備完了')

---
## 1. のこぎり波（Sawtooth）— 現在のベースライン

最も単純な近似。グロタルフローを「線形に上昇 → 瞬時に下降」とモデル化する。  
声帯の物理的な挙動とは大きくかけ離れているが、計算コストが極めて低い。

- **フロー**：鋸歯状（線形上昇）
- **微分**：定数 + インパルス（インパルス列が声道を励振）
- **スペクトル**：倍音が完全な $-6$ dB/oct で減衰（理論通りの倍音列）
- **問題**：開口・閉口の非対称性が全くなく、現実の声とは違うスペクトル傾斜

In [ ]:
def sawtooth_pulse(T0: int) -> np.ndarray:
    """のこぎり波グロタルフロー（1 周期）。0→1 の線形上昇。"""
    return np.linspace(0.0, 1.0, T0, endpoint=False).astype(np.float32)


def make_source_seq(
    pulse_fn,
    n_samples: int,
    f0:        float = 120.0,
    fs:        int   = FS,
    amplitude: float = 1.0,
    jitter:    float = 0.004,
    apply_radiation: bool = True,
) -> np.ndarray:
    """
    任意のパルス関数から周期的音源列を生成する汎用関数。

    Parameters
    ----------
    pulse_fn        : T0 → array（1 周期の波形を返す関数）
    apply_radiation : True の場合、口唇放射（一次差分）を適用する
    """
    T0  = int(fs / f0)
    out = np.zeros(n_samples, dtype=np.float64)
    pos = 0
    rng = np.random.default_rng(42)
    while pos < n_samples:
        Tc  = int(T0 * (1.0 + rng.uniform(-jitter, jitter)))
        Tc  = max(int(fs/400), min(int(fs/50), Tc))
        pulse = pulse_fn(Tc).astype(np.float64)
        end  = min(pos + Tc, n_samples)
        out[pos:end] += pulse[:end - pos]
        pos += Tc
    if apply_radiation:
        out = np.diff(out, prepend=out[0])   # 口唇放射：一次差分（+6 dB/oct）
    pk = np.max(np.abs(out))
    if pk > 1e-10:
        out = out / pk * amplitude
    return out.astype(np.float32)


# ── のこぎり波の波形・微分・スペクトルを確認 ────────────────────
T0_demo = int(FS / 150)   # f0 = 150 Hz
sw = sawtooth_pulse(T0_demo)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
t_ms = np.arange(T0_demo) / FS * 1000
axes[0].plot(t_ms, sw, color='#94a3b8')
axes[0].set(title='フロー', xlabel='時間 [ms]', ylabel='振幅')
axes[0].grid(True)

d = np.diff(sw, prepend=sw[0])
axes[1].plot(t_ms, d, color='#94a3b8')
axes[1].set(title='フロー微分（口唇放射）', xlabel='時間 [ms]')
axes[1].grid(True)

seq_saw = make_source_seq(sawtooth_pulse, 2*FS, f0=150)
f_w, P = signal.welch(seq_saw, fs=FS, nperseg=2048)
axes[2].plot(f_w, 10*np.log10(P+1e-12), color='#94a3b8')
axes[2].set(title='PSD', xlabel='周波数 [Hz]', xlim=(0,6000))
axes[2].grid(True)

fig.suptitle('のこぎり波（Sawtooth） — 現在のベースライン', fontsize=11)
plt.tight_layout()
plt.show()

print('のこぎり波（f₀=150Hz）:')
display(Audio(seq_saw, rate=FS))

---
## 2. Rosenberg C パルス

Rosenberg (1971) が提案した区分多項式によるグロタルフロー近似。  
**3 次エルミート補間（スムーズステップ）** を用いて開口・閉口フェーズを記述する。

$$
U_g(t) = \begin{cases}
3(t/T_p)^2 - 2(t/T_p)^3 & 0 \le t < T_p \quad \text{（開口 0→1）}\\
1 - \bigl[3(t'/T_n)^2 - 2(t'/T_n)^3\bigr] & T_p \le t < T_p+T_n \quad \text{（閉口 1→0、}t'=t-T_p\text{）}\\
0 & \text{その他（閉鎖フェーズ）}
\end{cases}
$$

| パラメータ | 意味 | 典型値 |
|------------|------|--------|
| $T_p/T_0$ | 開口フェーズ比率 | 0.35〜0.55 |
| $T_n/T_0$ | 閉口フェーズ比率 | 0.10〜0.20 |
| 閉鎖フェーズ | $1 - T_p/T_0 - T_n/T_0$ | 0.30〜0.50 |

$T_p > T_n$ の**非対称性**（ゆっくり開いて素早く閉まる）が人間の声帯の特徴。

In [ ]:
def rosenberg_pulse(T0: int, Tp_ratio: float = 0.45, Tn_ratio: float = 0.15) -> np.ndarray:
    """
    Rosenberg C グロタルフロー（1 周期）。
    3 次エルミート補間（smoothstep）で開口・閉口フェーズを記述する。

    Parameters
    ----------
    Tp_ratio : 開口フェーズの T0 に対する比率（0 < Tp < 1）
    Tn_ratio : 閉口フェーズの T0 に対する比率（Tp + Tn < 1）
    """
    T1 = max(2, int(Tp_ratio * T0))   # 開口フェーズ長
    T2 = max(2, int(Tn_ratio * T0))   # 閉口フェーズ長

    pulse = np.zeros(T0)

    # 開口フェーズ: smoothstep 0 → 1
    x = np.linspace(0.0, 1.0, T1, endpoint=False)
    pulse[:T1] = 3*x**2 - 2*x**3

    # 閉口フェーズ: smoothstep 1 → 0
    end2 = min(T1 + T2, T0)
    T2_act = end2 - T1
    if T2_act > 0:
        x = np.linspace(0.0, 1.0, T2_act, endpoint=False)
        pulse[T1:end2] = 1.0 - (3*x**2 - 2*x**3)

    # 閉鎖フェーズ: 0（初期化済み）
    return pulse.astype(np.float32)


# ── パラメータ比較：Tp 比率を変えた場合 ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
t_ms = np.arange(T0_demo) / FS * 1000

params = [
    (0.40, 0.12, '#38bdf8', 'Tp=40% Tn=12%（対称寄り）'),
    (0.50, 0.15, '#34d399', 'Tp=50% Tn=15%（標準）'),
    (0.60, 0.18, '#f472b6', 'Tp=60% Tn=18%（開放的）'),
]
for i, (tp, tn, col, label) in enumerate(params):
    p   = rosenberg_pulse(T0_demo, tp, tn)
    d   = np.diff(p.astype(np.float64), prepend=p[0])

    axes[0][i].plot(t_ms, p, color=col)
    axes[0][i].set(title=f'フロー\n{label}', xlabel='時間 [ms]', ylim=(-0.1, 1.1))
    axes[0][i].axvline(tp * T0_demo/FS*1000, color='#fbbf24', linewidth=0.8, linestyle='--')
    axes[0][i].grid(True)

    axes[1][i].plot(t_ms, d, color=col)
    axes[1][i].set(title='微分（→スパイクが声道を励振）', xlabel='時間 [ms]')
    axes[1][i].grid(True)

fig.suptitle('Rosenberg C パルス：Tp 比率の影響（黄破線 = 閉鎖開始）', fontsize=11)
plt.tight_layout()
plt.show()

# 試聴
for tp, tn, col, label in params:
    fn  = lambda T, tp=tp, tn=tn: rosenberg_pulse(T, tp, tn)
    seq = make_source_seq(fn, 2*FS, f0=150)
    print(f'Rosenberg C {label}:')
    display(Audio(seq, rate=FS))

---
## 3. LF モデル（Liljencrants-Fant, 1985）

音声合成で最も広く使われる物理ベースの声帯音源モデル。  
Pink Trombone や VocalTractLab でも採用されている。

### フロー微分の定義

LF モデルは **グロタルフロー微分 $dU_g/dt$** を直接定義する：

$$
\frac{dU_g}{dt} = \begin{cases}
E_0 \, e^{\alpha t} \sin(\omega_g t) & 0 \le t \le T_e \quad \text{（開口・閉口フェーズ）}\\
\dfrac{-e^{-\varepsilon(t-T_e)} + S}{\Delta} & T_e < t \le T_0 \quad \text{（帰還フェーズ）}
\end{cases}
$$

ここで $\omega_g = \pi / T_p$（開口ピーク時刻 $T_p$ で符号反転）。

### パラメータの意味

| パラメータ | 意味 | のどのつかえ |
|------------|------|--------------|
| $T_p$ | 微分の正ピーク時刻 | 開口の速さ |
| $T_e$ | 声帯閉鎖時刻（負スパイク）| 閉鎖の鋭さ |
| $T_a = 1/\varepsilon$ | 帰還時定数 | 息漏れ・気息声の量 |

### Rd パラメータ（声質制御）

Pink Trombone 流の **Rd（voice quality）** で上記パラメータをまとめて制御できる：

| Rd | 声質 | 特徴 |
|----|------|------|
| 0.5〜0.8 | 声門開鎖音・きしみ声 | 極めて鋭い閉鎖スパイク |
| 1.0〜1.7 | **通常の話し声** | バランスが取れている |
| 2.0〜2.7 | 息漏れ声・気息声 | スパイクが弱く息っぽい |

$$
R_a = -0.01 + 0.048\,R_d, \quad
R_k = 0.224 + 0.118\,R_d, \quad
R_g = \frac{R_k/4 \cdot (0.5 + 1.2\,R_k)}{0.11\,R_d - R_a(0.5 + 1.2\,R_k)}
$$

$$
T_p = \frac{T_0}{2 R_g}, \quad T_e = T_p(1 + R_k), \quad T_a = R_a \, T_0
$$

In [ ]:
def lf_params_from_Rd(Rd: float) -> dict:
    """
    Rd から LF モデルパラメータを計算する（Pink Trombone のアルゴリズム）。
    """
    Rd = float(np.clip(Rd, 0.5, 2.7))
    Ra = -0.01 + 0.048 * Rd
    Rk =  0.224 + 0.118 * Rd
    Rg = (Rk / 4) * (0.5 + 1.2 * Rk) / (0.11 * Rd - Ra * (0.5 + 1.2 * Rk))
    Tp = 1.0 / (2 * Rg)     # T0 に対する比率
    Te = Tp * (1 + Rk)       # T0 に対する比率
    Ta = Ra                   # T0 に対する比率
    return {'Rd': Rd, 'Ra': Ra, 'Rk': Rk, 'Rg': Rg, 'Tp': Tp, 'Te': Te, 'Ta': Ta}


def lf_pulse(T0: int, Rd: float = 1.5) -> np.ndarray:
    """
    LF モデルのグロタルフロー微分（1 周期）。
    Pink Trombone のアルゴリズムを Python に移植。

    Returns
    -------
    dUg : グロタルフロー微分（長さ T0 の配列）。
          フローは cumsum で求められる。
    """
    p   = lf_params_from_Rd(Rd)
    Tp  = max(2, int(p['Tp'] * T0))
    Te  = max(Tp + 1, int(p['Te'] * T0))
    Ta  = max(1.0, p['Ta'] * T0)

    epsilon = 1.0 / Ta
    shift   = np.exp(-epsilon * (T0 - Te))
    delta   = max(1.0 - shift, 1e-10)

    omega = np.pi / Tp
    s     = np.sin(omega * Te)
    if abs(s) < 1e-10:
        s = 1e-10

    # α を決定（数値的に）
    RHS = ((1.0 / epsilon) * (shift - 1.0) + (T0 - Te) * shift) / delta
    y   = -np.pi * s * RHS / (2.0 * Tp)
    y   = max(y, 1e-10)
    alpha = np.log(y) / (Tp / 2.0 - Te)
    E0    = -1.0 / (s * np.exp(alpha * Te))

    # 波形生成
    t    = np.arange(T0, dtype=np.float64)
    dudt = np.zeros(T0)

    # 開口〜閉口フェーズ（0 ≤ t ≤ Te）
    m_open   = t <= Te
    dudt[m_open] = E0 * np.exp(alpha * t[m_open]) * np.sin(omega * t[m_open])

    # 帰還フェーズ（Te < t ≤ T0）
    m_close  = t > Te
    tc       = t[m_close] - Te
    dudt[m_close] = (-np.exp(-epsilon * tc) + shift) / delta

    # 正規化
    pk = np.max(np.abs(dudt))
    if pk > 1e-10:
        dudt /= pk

    return dudt.astype(np.float32)


def lf_flow(T0: int, Rd: float = 1.5) -> np.ndarray:
    """LF グロタルフロー（微分を積分した値）。"""
    dudt = lf_pulse(T0, Rd).astype(np.float64)
    flow = np.cumsum(dudt)
    flow -= flow.mean()
    pk = np.max(np.abs(flow))
    if pk > 1e-10:
        flow /= pk
    return flow.astype(np.float32)


# ── LF モデルの波形を Rd ごとに確認 ─────────────────────────────
Rd_vals  = [0.6, 1.2, 1.8, 2.4]
Rd_labels= ['Rd=0.6\nきしみ声', 'Rd=1.2\nやや強め', 'Rd=1.8\n通常', 'Rd=2.4\n息漏れ']
Rd_colors= ['#ef4444', '#f59e0b', '#34d399', '#38bdf8']

fig, axes = plt.subplots(3, 4, figsize=(15, 8))

for i, (Rd, lbl, col) in enumerate(zip(Rd_vals, Rd_labels, Rd_colors)):
    flow = lf_flow(T0_demo, Rd)
    dudt = lf_pulse(T0_demo, Rd)
    p    = lf_params_from_Rd(Rd)
    Tp_s = int(p['Tp'] * T0_demo)
    Te_s = int(p['Te'] * T0_demo)

    # フロー
    axes[0][i].plot(t_ms, flow, color=col)
    axes[0][i].axvline(Tp_s/FS*1000, color='#fbbf24', linewidth=0.8, linestyle='--', label='Tp')
    axes[0][i].axvline(Te_s/FS*1000, color='#f472b6', linewidth=0.8, linestyle='--', label='Te')
    axes[0][i].set(title=lbl, ylabel='フロー')
    axes[0][i].legend(fontsize=7)
    axes[0][i].grid(True)

    # フロー微分
    axes[1][i].plot(t_ms, dudt, color=col)
    axes[1][i].axvline(Te_s/FS*1000, color='#f472b6', linewidth=0.8, linestyle='--')
    axes[1][i].set(ylabel='dU/dt（励振）')
    axes[1][i].grid(True)

    # PSD
    seq = make_source_seq(lambda T, Rd=Rd: lf_pulse(T, Rd),
                          2*FS, f0=150, apply_radiation=False)
    fw, P = signal.welch(seq, fs=FS, nperseg=2048)
    axes[2][i].plot(fw, 10*np.log10(P+1e-12), color=col)
    axes[2][i].set(ylabel='PSD [dB/Hz]', xlabel='周波数 [Hz]', xlim=(0,6000))
    axes[2][i].grid(True)

axes[0][0].set_ylabel('グロタルフロー')
axes[1][0].set_ylabel('フロー微分\n（声道励振）')
axes[2][0].set_ylabel('パワースペクトル\n[dB/Hz]')
fig.suptitle('LF モデル：Rd による声質変化（黄=Tp, ピンク=Te）', fontsize=11)
plt.tight_layout()
plt.show()

---
## 4. モデル間の比較

### 4-A. 波形・微分・スペクトルの並列比較

In [ ]:
models = [
    ('のこぎり波',       '#94a3b8', sawtooth_pulse,
     lambda T: sawtooth_pulse(T)),
    ('Rosenberg C\n(Tp=50%,Tn=15%)', '#fbbf24',
     lambda T: rosenberg_pulse(T),
     lambda T: rosenberg_pulse(T)),
    ('LF Rd=1.2\n（やや強め）', '#34d399',
     lambda T: lf_flow(T, 1.2),
     lambda T: lf_pulse(T, 1.2)),
    ('LF Rd=1.8\n（通常話し声）', '#38bdf8',
     lambda T: lf_flow(T, 1.8),
     lambda T: lf_pulse(T, 1.8)),
]

fig, axes = plt.subplots(3, 4, figsize=(15, 7))

for i, (label, col, flow_fn, excite_fn) in enumerate(models):
    flow  = flow_fn(T0_demo).astype(np.float64)
    excite = excite_fn(T0_demo).astype(np.float64)
    # のこぎり波の「励振」は微分
    if i == 0:
        excite = np.diff(flow, prepend=flow[0])
        pk = np.max(np.abs(excite))
        if pk > 1e-10: excite /= pk

    axes[0][i].plot(t_ms, flow, color=col)
    axes[0][i].set(title=label, ylabel='フロー')
    axes[0][i].grid(True)

    axes[1][i].plot(t_ms, excite, color=col)
    axes[1][i].set(ylabel='励振波形')
    axes[1][i].grid(True)

    seq = make_source_seq(excite_fn, 2*FS, f0=150, apply_radiation=False)
    fw, P = signal.welch(seq, fs=FS, nperseg=2048)
    axes[2][i].plot(fw, 10*np.log10(P+1e-12), color=col)
    axes[2][i].set(ylabel='PSD [dB/Hz]', xlabel='周波数 [Hz]', xlim=(0,6000))
    axes[2][i].grid(True)

axes[0][0].set_ylabel('グロタルフロー')
axes[1][0].set_ylabel('声道励振波形')
axes[2][0].set_ylabel('PSD [dB/Hz]')
fig.suptitle('声帯音源モデルの比較（行1=フロー, 行2=励振, 行3=スペクトル）',
             fontsize=11)
plt.tight_layout()
plt.savefig('../data/processed/analysis/glottal_models_comparison.png',
            dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('画像を data/processed/analysis/glottal_models_comparison.png に保存')

### 4-B. 単体音の試聴（母音 /a/ 形状 + 各音源）

In [ ]:
# 簡易的な声道フィルタ（/a/ の 3 フォルマント）
def resonator_sos(f, bw, fs=FS):
    r = np.exp(-np.pi * bw / fs)
    return signal.zpk2sos([], [r*np.exp(1j*2*np.pi*f/fs), r*np.exp(-1j*2*np.pi*f/fs)], 1.0)

def apply_vocal_tract(seq, formants, fs=FS):
    """フォルマントフィルタを適用する。"""
    out = seq.astype(np.float64)
    for f, bw in formants:
        out = signal.sosfilt(resonator_sos(f, bw, fs), out)
    pk = np.max(np.abs(out))
    if pk > 1e-10: out /= pk
    return out.astype(np.float32)

# /a/ のフォルマント（splab.net データから）
FORMANTS_A = [(750, 100), (1200, 120), (2600, 180)]

print('各音源モデルで合成した /a/（声道フィルタあり）:\n')

listen_models = [
    ('のこぎり波',       lambda T: sawtooth_pulse(T),           True),
    ('Rosenberg C',     lambda T: rosenberg_pulse(T, 0.50, 0.15), True),
    ('LF Rd=0.8（きしみ）', lambda T: lf_pulse(T, 0.8),          False),
    ('LF Rd=1.5（標準）',   lambda T: lf_pulse(T, 1.5),          False),
    ('LF Rd=2.2（息漏れ）', lambda T: lf_pulse(T, 2.2),          False),
]

for label, fn, apply_rad in listen_models:
    seq = make_source_seq(fn, int(1.5*FS), f0=150, apply_radiation=apply_rad)
    wav = apply_vocal_tract(seq, FORMANTS_A)
    print(f'{label}:')
    display(Audio(wav, rate=FS))

---
## 5. Rd パラメータの連続スイープ

`Rd` を低値（きしみ声）から高値（息漏れ声）まで変化させて  
声質の連続変化を試聴する。

In [ ]:
print('Rd スイープ（きしみ → 通常 → 息漏れ）\n')
for Rd in [0.6, 1.0, 1.5, 2.0, 2.5]:
    seq = make_source_seq(lambda T, Rd=Rd: lf_pulse(T, Rd),
                          int(1.5*FS), f0=150, apply_radiation=False)
    wav = apply_vocal_tract(seq, FORMANTS_A)
    print(f'Rd = {Rd}:')
    display(Audio(wav, rate=FS))

---
## 6. 合成パイプラインへの統合方法

現在の Phase 2・3 パイプラインでは `make_glottal_seq()` が  
「簡略LFパルス + 口唇放射（np.diff）」を行っていた。  
以下のように差し替えれば Rosenberg / LF モデルをそのまま使える。

In [ ]:
def make_glottal_seq_v2(
    n_samples:  int,
    f0:         float = 120.0,
    model:      str   = 'lf',       # 'sawtooth' | 'rosenberg' | 'lf'
    Rd:         float = 1.5,         # LF モデルの声質パラメータ
    Tp_ratio:   float = 0.50,        # Rosenberg の開口比
    Tn_ratio:   float = 0.15,        # Rosenberg の閉口比
    amplitude:  float = 1.0,
    jitter:     float = 0.004,
    fs:         int   = FS,
) -> np.ndarray:
    """
    選択したモデルでグロタル音源列を生成する。

    model='sawtooth'   : のこぎり波 + 口唇放射（後処理 diff）
    model='rosenberg'  : Rosenberg C フロー + 口唇放射
    model='lf'         : LF モデル（微分が直接出力、口唇放射は不要）
    """
    if model == 'sawtooth':
        fn  = lambda T: sawtooth_pulse(T)
        rad = True
    elif model == 'rosenberg':
        fn  = lambda T: rosenberg_pulse(T, Tp_ratio, Tn_ratio)
        rad = True
    elif model == 'lf':
        fn  = lambda T: lf_pulse(T, Rd)
        rad = False    # LF は微分が出力なので放射フィルタ不要
    else:
        raise ValueError(f'不明なモデル: {model}')

    return make_source_seq(fn, n_samples, f0=f0, fs=fs,
                           amplitude=amplitude, jitter=jitter,
                           apply_radiation=rad)


# 動作確認：同じ f0 で 3 モデルを合成
print('make_glottal_seq_v2() の動作確認（/a/ フォルマントあり）:\n')
for mdl, kw in [
        ('sawtooth',  {}),
        ('rosenberg', {'Tp_ratio': 0.50, 'Tn_ratio': 0.15}),
        ('lf',        {'Rd': 1.5}),
        ('lf',        {'Rd': 0.8}),
]:
    seq = make_glottal_seq_v2(int(1.5*FS), f0=150, model=mdl, **kw)
    wav = apply_vocal_tract(seq, FORMANTS_A)
    print(f'model={mdl} {kw}:')
    display(Audio(wav, rate=FS))

---
## まとめ：各モデルの特性と推奨用途

| モデル | 音質 | 計算コスト | 推奨用途 |
|--------|------|------------|----------|
| **のこぎり波** | ロボット的・倍音が多い | 最低 | テスト・プロトタイプ確認のみ |
| **Rosenberg C** | 自然な丸み・調整しやすい | 低 | 汎用音声合成・リアルタイム向き |
| **LF Rd=1.0〜1.7** | **最も自然** | 中 | **本番実装に推奨** |
| **LF Rd<1.0** | きしみ・声門閉鎖音 | 中 | 特殊声質表現 |
| **LF Rd>2.0** | 息漏れ・気息声 | 中 | 感情・声質バリエーション |

### 次のステップ

1. **Phase 3 パイプラインへの組み込み**  
   `make_glottal_seq_v2()` を使って `synth_fric_hybrid()` の声帯音源を差し替える

2. **Rd の動的制御**  
   発話中に Rd を変化させることで感情・アクセントの表現が可能

3. **LF の `lf_pulse()` を waveguide プロトタイプに移植**  
   JavaScript に翻訳して AudioWorklet 内で使用（Pink Trombone の実装が参考になる）

4. **実音声との照合**  
   EGG（電気声門計）で計測したグロタルフロー波形と LF モデルを比較し  
   適切な Rd 範囲を確認する